In [11]:
# !pip install scikit-posthocs

In [3]:
from scipy.stats import f_oneway, shapiro, kruskal, wilcoxon
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np

In [35]:
# MAE para cada fold de cada modelo
mae_mlp = [42.0543, 45.2809, 45.2541, 43.4433, 41.8344]
mae_rna = [45.5012, 52.3862, 39.7903, 30.8527, 27.1338]
mae_rna_tuning = [49.9603, 114.8611, 47.4836, 49.3284, 45.4279]

In [29]:
# Realizando o teste ANOVA
f_stat, p_value = f_oneway(mae_mlp, mae_rna, mae_rna_tuning)

print(f"Estatística F: {f_stat:.4f}")
print(f"Valor-p: {p_value:.4f}")

# Interpretação do resultado
if p_value < 0.05:
    print("Rejeitamos a hipótese nula. Pelo menos um modelo é significativamente diferente.")
else:
    print("Não rejeitamos a hipótese nula. Não há diferença significativa entre os modelos.")

Estatística F: 2.0735
Valor-p: 0.1685
Não rejeitamos a hipótese nula. Não há diferença significativa entre os modelos.


In [30]:
# Teste de Shapiro-Wilk para cada modelo
stat_mlp, p_mlp = shapiro(mae_mlp)
stat_rna, p_rna = shapiro(mae_rna)
stat_rna_tuning, p_rna_tuning = shapiro(mae_rna_tuning)

# Exibir resultados
print(f"MLP: Estatística W = {stat_mlp:.4f}, p-valor = {p_mlp:.4f}")
print(f"RNA: Estatística W = {stat_rna:.4f}, p-valor = {p_rna:.4f}")
print(f"RNA Tuning: Estatística W = {stat_rna_tuning:.4f}, p-valor = {p_rna_tuning:.4f}")

# Interpretação
alpha = 0.05  # Nível de significância
if p_mlp < alpha:
    print("MLP: Os dados NÃO seguem uma distribuição normal.")
else:
    print("MLP: Os dados seguem uma distribuição normal.")

if p_rna < alpha:
    print("RNA: Os dados NÃO seguem uma distribuição normal.")
else:
    print("RNA: Os dados seguem uma distribuição normal.")

if p_rna_tuning < alpha:
    print("RNA Tuning: Os dados NÃO seguem uma distribuição normal.")
else:
    print("RNA Tuning: Os dados seguem uma distribuição normal.")

MLP: Estatística W = 0.8460, p-valor = 0.1824
RNA: Estatística W = 0.9620, p-valor = 0.8219
RNA Tuning: Estatística W = 0.6098, p-valor = 0.0008
MLP: Os dados seguem uma distribuição normal.
RNA: Os dados seguem uma distribuição normal.
RNA Tuning: Os dados NÃO seguem uma distribuição normal.


In [31]:
# Teste de Kruskal-Wallis
h_stat, p_kruskal = kruskal(mae_mlp, mae_rna, mae_rna_tuning)
print(f"Estatística H: {h_stat:.4f}")
print(f"Valor-p: {p_kruskal:.4f}")

# Se p < 0.05, há diferença significativa → Aplicar teste de Dunn
if p_kruskal < 0.05:
    print("Há diferença significativa entre os modelos. Aplicando Dunn...")
    
    # Criar tabela para o teste de Dunn
    all_data = [mae_mlp, mae_rna, mae_rna_tuning]
    dunn_result = sp.posthoc_dunn(all_data, p_adjust="bonferroni")
    
    print(dunn_result)
else:
    print("Não há diferença significativa entre os modelos.")

Estatística H: 6.0000
Valor-p: 0.0498
Há diferença significativa entre os modelos. Aplicando Dunn...
          1         2         3
1  1.000000  1.000000  0.101685
2  1.000000  1.000000  0.101685
3  0.101685  0.101685  1.000000


In [32]:
stat, p_value = wilcoxon(mae_mlp, mae_rna)

print(f"Estatística de Wilcoxon: {stat:.4f}")
print(f"Valor-p: {p_value:.4f}")

# Interpretação do resultado
if p_value < 0.05:
    print("Há diferença significativa entre os modelos.")
else:
    print("Não há diferença significativa entre os modelos.")

Estatística de Wilcoxon: 4.0000
Valor-p: 0.4375
Não há diferença significativa entre os modelos.


In [33]:
# Comparação entre os modelos
print("MLP vs RNA:")
print(wilcoxon(mae_mlp, mae_rna))

print("\nMLP vs RNA Tuning:")
print(wilcoxon(mae_mlp, mae_rna_tuning))

print("\nRNA vs RNA Tuning:")
print(wilcoxon(mae_rna, mae_rna_tuning))

MLP vs RNA:
WilcoxonResult(statistic=4.0, pvalue=0.4375)

MLP vs RNA Tuning:
WilcoxonResult(statistic=0.0, pvalue=0.0625)

RNA vs RNA Tuning:
WilcoxonResult(statistic=0.0, pvalue=0.0625)


In [34]:
import pandas as pd
import scipy.stats as stats
import statsmodels.sandbox.stats.multicomp as multi
import numpy as np

data = {
    'MLP': [42.0543, 45.2809, 45.2541, 43.4433, 41.8344],
    'RNA': [45.5012, 52.3862, 39.7903, 30.8527, 27.1338],
    'RNA Tuning': [49.9603, 114.8611, 47.4836, 49.3284, 45.4279]
}

df = pd.DataFrame(data)

# Teste de Friedman
stat, p = stats.friedmanchisquare(df['MLP'], df['RNA'], df['RNA Tuning'])
print(f"Friedman: stat={stat:.3f}, p={p:.4f}")

# Testes de Wilcoxon Pairwise
print("\nWilcoxon Pairwise:")
for col1 in df.columns:
    for col2 in df.columns:
        if col1 != col2:
            stat, p = stats.wilcoxon(df[col1], df[col2])
            print(f"{col1} vs {col2}: stat={stat:.3f}, p={p:.4f}")

# Análise Descritiva
print("\nAnálise Descritiva:")
print(df.mean())
print(df.std())

# Teste de Nemenyi (post-hoc para Friedman) - Corrigido
print("\nNemenyi Test:")
data_values = df.values.flatten()  # Flatten the DataFrame values
groups = np.repeat(df.columns, len(df))  # Create groups for each value
m_comp = multi.MultiComparison(data_values, groups)
result = m_comp.tukeyhsd()
print(result)

Friedman: stat=7.600, p=0.0224

Wilcoxon Pairwise:
MLP vs RNA: stat=4.000, p=0.4375
MLP vs RNA Tuning: stat=0.000, p=0.0625
RNA vs MLP: stat=4.000, p=0.4375
RNA vs RNA Tuning: stat=0.000, p=0.0625
RNA Tuning vs MLP: stat=0.000, p=0.0625
RNA Tuning vs RNA: stat=0.000, p=0.0625

Análise Descritiva:
MLP           43.57340
RNA           39.13284
RNA Tuning    61.41226
dtype: float64
MLP            1.665009
RNA           10.358300
RNA Tuning    29.930974
dtype: float64

Nemenyi Test:
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
group1   group2   meandiff p-adj   lower    upper  reject
---------------------------------------------------------
   MLP        RNA  11.1299 0.6441 -21.4726 43.7324  False
   MLP RNA Tuning  -8.1211 0.7879 -40.7237 24.4814  False
   RNA RNA Tuning  -19.251 0.2931 -51.8536 13.3515  False
---------------------------------------------------------
